In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
df = pd.read_csv("../UH_2023/UH_2023_TRAIN.txt", sep = "|")

In [ ]:
df = df.astype({"CAMPA?A": str, "ID_FINCA": str, "ID_ZONA": str, "ID_ESTACION": str, "VARIEDAD": str, "MODO": str, "TIPO": str, "COLOR": str})

In [ ]:
df

In [ ]:
import category_encoders as ce
encoder = ce.OrdinalEncoder(cols=["ID_FINCA", "ID_ZONA", "ID_ESTACION", "VARIEDAD", "COLOR", "TIPO", "MODO"])
df = encoder.fit_transform(df)

In [ ]:
df

In [ ]:
encoder.inverse_transform(df)

In [ ]:
df_alt = df.drop(columns = ["SUPERFICIE", "PRODUCCION"])
X_df_alt = df_alt.drop(columns = ["ALTITUD", "CAMPA?A"])
y_df_alt = df_alt["ALTITUD"]

X_df_alt_train = X_df_alt[df["ALTITUD"].isna() == False]
y_df_alt_train = y_df_alt[df["ALTITUD"].isna() == False]

X_to_predict = X_df_alt[df["ALTITUD"].isna() == True]

In [ ]:
y_df_alt_train

In [ ]:
X_df_alt_train_n390 = X_df_alt_train[y_df_alt_train != "390"]
y_df_alt_train_n390 = y_df_alt_train[y_df_alt_train != "390"]

In [ ]:
X_df_alt_train[y_df_alt_train == "390"]

In [ ]:
y_df_alt_train_n390_enc = pd.Series(LabelEncoder().fit_transform(y_df_alt_train_n390))

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

def grid(X,y):
    gsc = GridSearchCV(
                estimator=xgb.XGBClassifier(),
                param_grid={"n_estimators": [300, 500, 1000],
                            "learning_rate": [0.15],
                            "max_depth": [40, 60, 100],
                            "min_child_weight": [1, 2],
                            "gamma":[0.0],
                            "colsample_bytree":[0.4],},
                cv=2, scoring=["r2", "neg_root_mean_squared_error"], refit = "r2", verbose=0, n_jobs=-1)
    gsc.fit(X,y)
    return gsc

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

def grid2(X,y):
    gsc = GridSearchCV(
                estimator=xgb.XGBClassifier(),
                param_grid={"n_estimators": [300],
                            "learning_rate": [0.15],
                            "max_depth": [40],
                            "min_child_weight": [1],
                            "gamma":[0.0],
                            "colsample_bytree":[0.4],},
                cv=2, scoring=["r2", "neg_root_mean_squared_error"], refit = "r2", verbose=0, n_jobs=-1)
    gsc.fit(X,y)
    return gsc

In [ ]:
alt_grid = grid2(X_df_alt_train_n390, y_df_alt_train_n390_enc)

In [ ]:
params_model = alt_grid.best_params_

In [ ]:
print(params_model)

In [ ]:
label_enc = LabelEncoder().fit(y_df_alt_train)
y_df_alt_train_enc = pd.Series(label_enc.transform(y_df_alt_train))

In [ ]:
model_alt_grid = xgb.XGBClassifier(**params_model)
model_alt_grid.fit(X_df_alt_train,y_df_alt_train_enc)
predicted = model_alt_grid.predict(X_to_predict)

In [ ]:
df2 = pd.read_csv("../UH_2023/UH_2023_TRAIN.txt", sep = "|")
df2 = df.astype({"CAMPA?A": str, "ID_FINCA": str, "ID_ZONA": str, "ID_ESTACION": str, "VARIEDAD": str, "MODO": str, "TIPO": str, "COLOR": str})

In [ ]:
predicted2 = pd.Series(label_enc.inverse_transform(predicted))
predicted2.index = df2[df2["ALTITUD"].isna() == True].index

In [ ]:
df2.fillna({"ALTITUD": predicted2}, inplace=True)

In [ ]:
df2

In [ ]:
def fun(x, mode): 
    altitud = str(x).split('-')
    if altitud[0] == "nan":
        if mode == "nan_1":
            return -1
        return np.nan
    if mode == "nan_1":
            return x
    if mode == "min":
        return int(altitud[0])
    if mode == "diff":
        if len(altitud) == 1:
            return 0
        else:
            return int(altitud[1])-int(altitud[0])

df2["ALTITUD_MIN"] = df2["ALTITUD"].apply(fun, args=("min",))
df2["ALTITUD_DIF"] = df2["ALTITUD"].apply(fun, args=("diff",))
df2["ALTITUD"] = df2["ALTITUD"].apply(fun, args=("nan_1",))

In [ ]:
df2.loc[df["MODO"]=="1", "MODO"]  = "0"
df2.loc[df["MODO"]=="2", "MODO"]  = "1"

In [ ]:
df2

In [ ]:
df_new = df2

### Dataset 1 - Tot

In [ ]:
df_new.to_csv("../DATASETS_TRACTATS/TOT_altitud.csv", index=False)

### Dataset 2 - Tot menys 14-15

In [ ]:
df_new2 = df_new.copy()
rows_to_drop = ["14", "15"]
df_new2 = df_new2[df_new2["CAMPA?A"].isin(rows_to_drop) != True]

In [ ]:
df_new2.to_csv("../DATASETS_TRACTATS/TOT_menys14_15_altitud.csv", index=False)

### Dataset 3 - Eliminant files de 14-15 on ID_FINCA_ZONA i VARIEDAD no hi s?n en el 22

In [ ]:
df_new3 = df_new.copy()
df_new3["ID_FINCA_ZONA"] = df_new3["ID_FINCA"] + "_"+ df_new3["ID_ZONA"]

In [ ]:
rows_not_to_drop_id_finca_zona = df_new3[df_new3["CAMPA?A"] == "22"]["ID_FINCA_ZONA"].unique()
rows_not_to_drop_varietat = df_new3[df_new3["CAMPA?A"] == "22"]["VARIEDAD"].unique
years_to_look = ["14", "15"]

In [ ]:
df_new3 = df_new3[(df_new3["CAMPA?A"].isin(years_to_look) == False)|
                  ((df_new3["CAMPA?A"].isin(years_to_look) == True)&
                  (df_new3["ID_FINCA_ZONA"].isin(rows_not_to_drop_id_finca_zona) == True)&
                   (df_new3["ID_FINCA_ZONA"].isin(rows_not_to_drop_id_finca_zona) == True))]

In [ ]:
df_new3 = df_new3.drop(columns= "ID_FINCA_ZONA")

In [ ]:
df_new3.to_csv("../DATASETS_TRACTATS/TOT_fzv_1415_altitud.csv", index=False)

### Dataset 4 - Eliminant files de 14-21 on ID_FINCA_ZONA no hi s?n en el 22

In [ ]:
df_new4 = df_new.copy()
df_new4["ID_FINCA_ZONA"] = df_new4["ID_FINCA"] + "_"+ df_new4["ID_ZONA"]

In [ ]:
rows_not_to_drop_id_finca_zona = df_new4[df_new4["CAMPA?A"] == "22"]["ID_FINCA_ZONA"].unique()
rows_not_to_drop_varietat = df_new4[df_new4["CAMPA?A"] == "22"]["VARIEDAD"].unique
years_to_look = ["14", "15", "16", "17", "18", "19", "20", "21"]
df_new4 = df_new4[(df_new4["CAMPA?A"].isin(years_to_look) == False)|
                  ((df_new4["CAMPA?A"].isin(years_to_look) == True)&
                  (df_new4["ID_FINCA_ZONA"].isin(rows_not_to_drop_id_finca_zona) == True)&
                   (df_new4["ID_FINCA_ZONA"].isin(rows_not_to_drop_id_finca_zona) == True))]

In [ ]:
df_new4 = df_new4.drop(columns= "ID_FINCA_ZONA")

In [ ]:
df_new4.to_csv("../DATASETS_TRACTATS/TOT_fzv_1421_altitud.csv", index=False)

### Dataset 5 - Eliminant files de 14-21 on ID_FINCA_ZONA no hi s?n en el 22

In [ ]:
df_new5 = df_new.copy()

In [ ]:
df_16_21 = df_new[df_new["CAMPA?A"].isin(["14", "15", "22"]) == False]
df_22 = df_new[df_new["CAMPA?A"].isin(["22"]) == True]

In [ ]:
id_finca = set(df_22["ID_FINCA"]) - set(df_16_21["ID_FINCA"])
id_zona = set(df_22["ID_ZONA"]) - set(df_16_21["ID_ZONA"])
id_estacion = set(df_22["ID_ESTACION"]) - set(df_16_21["ID_ESTACION"])
id_color = set(df_22["COLOR"]) - set(df_16_21["COLOR"])
id_tipo = set(df_22["TIPO"]) - set(df_16_21["TIPO"])
id_modo = set(df_22["MODO"]) - set(df_16_21["MODO"])

In [ ]:
df_new5[df_new5["ID_ZONA"].isin(id_zona)==True]

In [ ]:
df_new5[df_new5["ID_FINCA"].isin(id_finca)==True]

In [ ]:
df_new5[df_new5["SUPERFICIE"] == 2.2643]

In [ ]:
df_new5[df_new5["SUPERFICIE"] == 7.5143]

In [ ]:
dd = encoder.inverse_transform(df_new5.drop(columns = ["ALTITUD_MIN", "ALTITUD_DIF"]))

In [ ]:
df_new5

In [ ]:
encoder.inverse_transform(df_new5.drop(columns = ["ALTITUD_MIN", "ALTITUD_DIF"]))